## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# CNN applied to videos - part 2: tracking a barbell

In this notebook, we use an object detection model to follow the **barbell** during a weightlifting exercise, and we compute the **bar height and bar velocity**. Bar velocity is the main measure of *velocity-based training*: coaches use it to choose the load and to see fatigue.

What you will learn:
- how to apply a CNN to each video frame
- how to clean the detections with simple physical rules
- how to convert pixels to meters
- how to compute a velocity from a noisy position signal

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.signal import medfilt, savgol_filter, find_peaks

def show_detections(img, results, names):
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    for box, c in zip(results[0].boxes.xyxy.cpu().numpy(), results[0].boxes.cls.cpu().numpy()):
        x1, y1, x2, y2 = box
        plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='red', linewidth=2))
        plt.text(x1, y1, names[int(c)], color='red', fontsize=11)
    plt.axis('off'); plt.show()

First, we try the COCO detection model of 07.01 on one image of a lifter.

In [ ]:
img = Image.open('data/lifter.png').convert('RGB')
model = YOLO("../models/yolo26s.pt")
show_detections(img, model(img, verbose=False), model.names)

The model did a great job finding people in the image, but the model did not find the weights... 

It turns out that the weights are not part of the COCO categories...


### Model Fine-tuning 

To solve this problem, I collected a dataset of 1,000 images with weights, with the position of the weight in each image (a box).

<div style="text-align:center">
<img src="images/sampleBarbell.png" alt="plot" width="400" />
</div>

I used these images to **fine-tune** the detection model (as in 06.03). The training took about 20 minutes on HiPerGator (UF's supercomputer, with GPUs). On my personal computer, I stopped it after about 20 hours.


Let us check if the new model works.

In [ ]:
barbellmodel = YOLO("../models/barbelldetector.pt")
show_detections(img, barbellmodel(img, verbose=False), barbellmodel.names)

The model detected the barbell, but also other round objects. It learned to detect round shapes that look like weight plates.

A simple solution is to look only at the **region of interest**, where the barbell can be. Here, the lifter is in the center of the image, so we keep the columns 600 to 1500. We crop the PIL image (`crop((left, top, right, bottom))`), so the colors stay correct.

In [ ]:
W, H = img.size
print('Image size:', W, 'x', H)
img_crop = img.crop((600, 0, 1500, H))
show_detections(img_crop, barbellmodel(img_crop, verbose=False), barbellmodel.names)

Now we do this for every frame of a video.

In [ ]:
from IPython.display import Video, display
display(Video("data/lift_one.mp4", embed=True, width=480))

## Step 1 - Detect the barbell in each frame

We read the video with OpenCV (07.02a), crop the region of interest, and keep **all** the detections of each frame (the box and the confidence). We give the frame to YOLO directly: YOLO expects the BGR order of OpenCV.

In [ ]:
def detect_barbell(video_path, x_min=600, x_max=1500):
    video = cv2.VideoCapture(video_path)
    fps = video.get(cv2.CAP_PROP_FPS)
    detections = []          # one array per frame: rows = [x1, y1, x2, y2, confidence]
    while True:
        valid, frame = video.read()
        if not valid:
            break
        r = barbellmodel(frame[:, x_min:x_max], verbose=False)[0]
        if len(r.boxes) > 0:
            detections.append(np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()])
        else:
            detections.append(np.zeros((0, 5)))
    video.release()
    return detections, fps

detections, fps = detect_barbell("data/lift_one.mp4")
print(f'{len(detections)} frames, {fps} frames/s')
print('Detections per frame:', np.bincount([len(d) for d in detections]))

Most frames have more than one detection. We need to choose the correct one.

## Step 2 - Clean the detections with physical rules

We use what we know about the object:
1. **The plate has a fixed size.** A standard competition plate has a diameter of **0.45 m**, so its box always has about the same height in pixels. We remove boxes with a height more than 25% different from the typical plate height.
2. **The barbell cannot jump.** Between two frames (0.04 s), the bar moves only a few centimeters. If more than one box is left, we keep the one closest to the position in the previous frame.

Frames with no valid box get NaN, and we fill them by interpolation.

In [ ]:
def clean_track(detections):
    heights = np.concatenate([d[:, 3] - d[:, 1] for d in detections])
    plate_px = np.median(heights[heights > 150])          # typical height of a plate box (the large boxes)
    cx, cy, prev = [], [], None
    for d in detections:
        h = d[:, 3] - d[:, 1]
        ok = d[np.abs(h - plate_px) < 0.25 * plate_px]     # rule 1: plate size
        if len(ok) == 0:
            cx.append(np.nan); cy.append(np.nan); continue
        centers = np.c_[(ok[:, 0] + ok[:, 2]) / 2, (ok[:, 1] + ok[:, 3]) / 2]
        k = np.argmax(ok[:, 4]) if prev is None else np.argmin(np.linalg.norm(centers - prev, axis=1))   # rule 2
        prev = centers[k]
        cx.append(prev[0]); cy.append(prev[1])
    return np.array(cx), np.array(cy), plate_px

def fill_gaps(x):
    x = x.copy(); ok = np.isfinite(x)
    x[~ok] = np.interp(np.flatnonzero(~ok), np.flatnonzero(ok), x[ok])
    return x

cx, cy, plate_px = clean_track(detections)
print(f'Plate height: {plate_px:.0f} pixels | frames with no valid box: {np.isnan(cy).sum()}')

## Step 3 - From pixels to meters

The plate diameter is 0.45 m and its box is `plate_px` pixels high, so one pixel is `0.45 / plate_px` meters. Remember that in image coordinates **y increases downwards**, so the bar height is `-(y - y_start)`.

This conversion is correct only for movement in the plane of the plate (the plane parallel to the camera), and only if the camera does not move.

In [ ]:
m_per_px = 0.45 / plate_px
t = np.arange(len(cy)) / fps
height_raw = -(fill_gaps(cy) - fill_gaps(cy)[0]) * m_per_px      # bar height from the start position, in m
horizontal_raw = (fill_gaps(cx) - fill_gaps(cx)[0]) * m_per_px  # horizontal displacement, in m
print(f'1 pixel = {1000 * m_per_px:.1f} mm')

plt.figure(figsize=(10, 4))
plt.plot(t, height_raw)
plt.xlabel('Time (s)'); plt.ylabel('Bar height (m)'); plt.title('Bar height (raw)')
plt.show()

## Step 4 - Smooth the signal and compute the velocity

A derivative makes the noise much larger, so we smooth first: a **median filter** (lecture 03.03) removes single-frame errors, then a Savitzky–Golay filter smooths the signal and computes the derivative (the velocity).

In [ ]:
def smooth_and_velocity(x, fps):
    x_s = savgol_filter(medfilt(x, 5), window_length=9, polyorder=2)
    v = savgol_filter(x_s, window_length=9, polyorder=2, deriv=1, delta=1 / fps)
    return x_s, v

height, velocity = smooth_and_velocity(height_raw, fps)
horizontal, _ = smooth_and_velocity(horizontal_raw, fps)
peak = int(np.argmax(velocity))

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(t, height_raw, alpha=0.4, label='raw'); ax[0].plot(t, height, label='smoothed')
ax[0].set_ylabel('Bar height (m)'); ax[0].legend()
ax[1].plot(t, velocity, color='tab:orange'); ax[1].axhline(0, color='gray', linewidth=0.8)
ax[1].plot(t[peak], velocity[peak], 'x', color='red', markersize=10)
ax[1].set_ylabel('Vertical velocity (m/s)'); ax[1].set_xlabel('Time (s)')
plt.show()
print(f'Peak upward velocity: {velocity[peak]:.2f} m/s at {t[peak]:.2f} s | maximum height: {height.max():.2f} m')

In this lift, the bar accelerates during the pull and reaches its peak velocity, then it slows down and drops a little while the lifter moves under the bar (the catch), and then rises again when the lifter stands up.

## The bar path

We can also plot the path of the bar in the vertical plane (horizontal vs vertical displacement, in meters). Coaches use the bar path to check the technique.

In [ ]:
plt.figure(figsize=(5, 7))
plt.plot(horizontal, height, linewidth=2)
plt.scatter(horizontal[0], height[0], c='green', s=60, label='Start', zorder=3)
plt.scatter(horizontal[-1], height[-1], c='red', s=60, label='End', zorder=3)
plt.xlabel('Horizontal displacement (m)'); plt.ylabel('Bar height (m)')
plt.axis('equal'); plt.grid(alpha=0.3); plt.legend(); plt.title('Bar path')
plt.show()

## Activity

Do the same analysis for the video `data/lift_two.mp4`:
1. Detect the barbell, clean the track, and convert to meters.
2. Plot the bar height and the velocity. How many lifts (upward movements) do you see? What is the peak velocity of each one?
3. Remove rule 1 (the plate size) from `clean_track` and repeat. What happens to the signal? Why?

## Summary

- A CNN detects the object in each frame; the positions over time make a **signal**.
- Detections contain errors. Simple **physical rules** (a fixed size, no jumps) remove most of them.
- A known size in the image (the plate) converts pixels to meters, but only in the plane of that object.
- Smooth the position **before** computing a velocity.